# Import Library

In [1]:
import pandas as pd
import numpy as np
import csv
import openai
from openai import OpenAI
from tqdm import tqdm
import time

C:\Users\armin\AppData\Local\Temp\ipykernel_14896\2271314348.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# API Key

In [ ]:
# Your OpenAI API key
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="****",
)

# Reading File

In [ ]:
# 📂 خواندن جملات از فایل اکسل
df = pd.read_csv("file.csv")  # فایل ورودی باید یک ستون به نام "Sentence" داشته باشد

In [ ]:
sentences = df['Sentence']

In [ ]:
MODEL = None
MAX_TOKEN = None

# Score Sentences

In [98]:
def get_gpt_score(sentence):

    prompt = f"""
    شما باید خلاقیت یک جمله را بر اساس ۴ معیار زیر ارزیابی کنید.
    لطفاً برای هر معیار یک عدد بین **1 تا 3** بدهید. **فقط عدد را بدهید و هیچ توضیحی اضافه نکنید.
    کمترین نمره عدد 1 و بیشترین نمره عدد 3 است**


    ۱. **آیا جمله خلاقانه است و از کلیشه های رایج در زبان فارسی فاصله دارد؟**
    - ۱: جمله کاملاً کلیشه‌ای و بدون نوآوری است.
    - ۲: جمله نسبتاً خلاقانه است اما هنوز مشابه جملات رایج است.
    - ۳: جمله نوآورانه و غیرکلیشه‌ای است.

    ۲. **تنوع ایده‌های مورد استفاده در جمله به چه صورت است؟*
    - ۱: جمله ۱ ایده دارد.
    - ۲: جمله دارای ۲ ایده است.
    - ۳: ایده جمله بسیار جالب و با تعداد ایده‌ها ۳ یا بیشتر است

    ۳. **آیا جمله حس خاصی مانند غم، امید، شادی و غیره را منتقل میکند؟*
    - ۱: جمله حس خفیفی دارد و ضعیف است
    - ۲: جمله احساس مشخصی را منتقل میکند اما میتوان آن را تقویت کرد .
    - ۳: جمله به شدت احساسی و تاثیرگذار است.

    ۴. **تنوع کلمات و توصیف‌ها به چه صورت است و آیا جمله وارد جزییات شده است؟*
    - ۱: تنوع کلمات پایین و توصیفات ضعیف است.
    - ۲: تنوع کلمات متوسط و جمله تا حدی توصیفات خوبی دارد.
    - ۳: از کلمات متنوع استفاده شده و جمله توصیف جالبی دارد

    **مهم:** لطفاً برای هر سوال یک امتیاز عددی بین 1 تا 3 بدهید و فقط عدد بنویسید (مثلاً 1، 2 یا 3). هیچ توضیحی ندهید و از اعشاری (مانند 2.5 یا 1.33) استفاده نکنید.

    **مهم:** فقط در این فرمت پاسخ دهید و هیچ توضیحی اضافه نکنید:
    خلاقیت: X
    تنوع ایده: X
    احساس: X
    تنوع کلمات: X


    ### حالا جمله زیر را ارزیابی کنید:
    جمله: "{sentence}"
    """
    criteria = ["خلاقیت", "تنوع ایده", "احساس","تنوع کلمات"]
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "شما یک کارشناس در ارزیابی خلاقیت متون ادبی هستید."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=MAX_TOKEN,
        )
        # print(response)
        generated_text = response.choices[0].message.content.strip().lower()
        print(generated_text)

        # استخراج نمرات از پاسخ
        scores = {}
        for line in generated_text.split("\n"):
            if ":" in line:
                key, value = line.split(":")
                key, value = key.strip(), value.strip()

                if key in criteria:
                    try:
                        value = int(value)
                        if 1 <= value <= 3:
                            scores[key] = value
                        else:
                            scores[key] = None
                    except ValueError:
                        scores[key] = None


        # بررسی اینکه همه معیارها مقدار معتبر دارند
        if len(scores) != 4:
            return {crit: None for crit in criteria}

        return scores

    except Exception as e:
        print(f"❌ خطا در پردازش جمله: {sentence}\n🔹 خطا: {str(e)}")
        return {crit: None for crit in criteria}


In [ ]:
# 🚀 پردازش جملات و دریافت امتیازها از GPT-4
scores_list = []
criteria = ["خلاقیت", "تنوع ایده", "احساس","تنوع کلمات"]
for sentence in tqdm(sentences, desc=f"Scoring Sentences with {MODEL}"):
    print(sentence)
    scores = get_gpt_score(sentence)
    scores["sentence"] = sentence

    scores_list.append(scores)
    time.sleep(1)  # جلوگیری از نرخ بالای درخواست‌ها (API Rate Limit)

# تبدیل لیست امتیازات به یک DataFrame برای تجزیه و تحلیل
dataset = pd.DataFrame(scores_list)


In [ ]:
dataset.to_csv('submission.csv', index=False)

# Score Creativity

In [ ]:
def get_gpt_score(sentence):

    prompt = f"""
    لطفاً خلاقیت «عنوان و متن ادبی» زیر را ارزیابی کنید.

    فقط یک عدد بین **1 تا 3** بدهید:

    1) اصلاً خلاقانه نیست  
    2) کمی خلاقانه است  
    3) کاملاً خلاقانه است  

    **مهم:** فقط عدد را بدهید و هیچ توضیحی اضافه نکنید.  
    **مهم:** فقط خروجی را در این قالب بدهید:  
    خلاقیت: X

    جمله: "{sentence}"
    """

    criteria = ["خلاقیت"]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "شما یک کارشناس ارزیابی خلاقیت متون ادبی هستید."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=MAX_TOKEN,
        )

        generated_text = response.choices[0].message.content.strip()
        print(generated_text)

        # استخراج نمره
        scores = {}
        for line in generated_text.split("\n"):
            if ":" in line:
                key, value = line.split(":")
                key, value = key.strip(), value.strip()

                if key in criteria:
                    try:
                        value = int(value)
                        if 1 <= value <= 3:
                            scores[key] = value
                        else:
                            scores[key] = None
                    except:
                        scores[key] = None

        # اگر نمره معتبر نبود
        if len(scores) != 1:
            return {crit: None for crit in criteria}

        return scores

    except Exception as e:
        print(f"❌ خطا در پردازش جمله: {sentence}\n🔹 خطا: {str(e)}")
        return {crit: None for crit in criteria}


In [ ]:
scores_list = []
for sentence in tqdm(sentences, desc=f"Scoring Sentences with {MODEL}"):
    print(sentence)

    # گرفتن نمره از مدل
    score_dict = get_gpt_score(sentence)   # برمی‌گرداند {'خلاقیت': 2}
    score_value = score_dict.get("خلاقیت")  # فقط عدد 2 را بگیر

    scores_list.append({
        "sentence": sentence,
        "Creativity": score_value
    })

    time.sleep(1)  # جلوگیری از Rate Limit

# تبدیل لیست به DataFrame
creativity = pd.DataFrame(scores_list)


# Score Attractivness

In [ ]:
def get_gpt_score_attractiveness(sentence):

    prompt = f"""
    لطفاً جذابیت «متن ادبی و سبک آن» را ارزیابی کنید.

    فقط یک عدد بین **1 تا 3** بدهید:

    آیا متن ادبی و سبک آن را جذاب میبینید و شما را علاقه مند میکند؟
    
    1) متن و سبک را اصلاً جذاب نمیبینم و هیچ علاقه‌ای ندارم  
    2) متن و سبک را کمی جذاب میبینم و تا حدودی علاقه‌مند هستم  
    3) متن و سبک را کاملاً جذاب میبینم و بسیار به این متن علاقه دارم  

    **مهم:** فقط عدد را بدهید و هیچ توضیحی اضافه نکنید.  
    **مهم:** فقط خروجی را در این قالب بدهید:  
    Attractiveness: X

    جمله: "{sentence}"
    """

    criteria = ["Attractiveness"]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "شما یک کارشناس ارزیابی جذابیت متون ادبی هستید."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=MAX_TOKEN,
        )

        generated_text = response.choices[0].message.content.strip()
        print(generated_text)

        # استخراج نمره
        scores = {}
        for line in generated_text.split("\n"):
            if ":" in line:
                key, value = line.split(":")
                key, value = key.strip(), value.strip()

                if key in criteria:
                    try:
                        value = int(value)
                        if 1 <= value <= 3:
                            scores[key] = value
                        else:
                            scores[key] = None
                    except:
                        scores[key] = None

        # اگر نمره معتبر نبود
        if len(scores) != 1:
            return {crit: None for crit in criteria}

        return scores

    except Exception as e:
        print(f"❌ خطا در پردازش جمله: {sentence}\n🔹 خطا: {str(e)}")
        return {crit: None for crit in criteria}


In [ ]:
# 🚀 پردازش جملات و دریافت امتیاز جذابیت از GPT-4
scores_list = []
for sentence in tqdm(sentences, desc=f"Scoring Sentences with {MODEL}"):
    print(sentence)

    # گرفتن نمره از مدل
    score_dict = get_gpt_score_attractiveness(sentence)   # برمی‌گرداند {'Attractiveness': 2}
    score_value = score_dict.get("Attractiveness")        # فقط عدد 2 را بگیر

    scores_list.append({
        "sentence": sentence,
        "Attractiveness": score_value
    })

    time.sleep(1)  # جلوگیری از Rate Limit

# تبدیل لیست به DataFrame
attractiveness = pd.DataFrame(scores_list)


# Merge Attractivenss and Creativity

In [ ]:
# ادغام دیتافریم‌ها روی ستون sentence
merged_df = pd.merge(creativity, attractiveness, on="sentence")

In [ ]:
merged_df.to_csv("submission_att_cre.csv", index=False)